In [1]:
import nltk
from nltk.corpus import movie_reviews, stopwords
import re
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
nltk.download('movie_reviews')
nltk.download('stopwords')

#wordnet lemmatizer
#punkt for tokenization
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/abhishekgrover/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/abhishekgrover/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/abhishekgrover/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/abhishekgrover/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/abhishekgrover/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
stop_words=set(stopwords.words('english'))
doc=[" ".join(movie_reviews.words(fid)) for fid in movie_reviews.fileids()]
label=[movie_reviews.categories(id)[0] for id in movie_reviews.fileids()]

In [4]:
lemmatizer=WordNetLemmatizer()

def clean_texta(text):
    text=text.lower()#Normalization
    text=re.sub(r'[^a-z\s]','',text)
    tokens=nltk.word_tokenize(text)
    tokens=[lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(tokens)

doc=[clean_texta(" ".join(movie_reviews.words(fid))) for fid in movie_reviews.fileids()]

In [5]:
X_train,X_test,Y_train,Y_test=train_test_split(doc,label,test_size=0.2,random_state=5,
                                               stratify=label)
v=CountVectorizer(max_features=5000)
X_trainv=v.fit_transform(X_train)
X_testv=v.transform(X_test)
X_trainv=X_trainv.toarray()
X_testv=X_testv.toarray()
Y_train=np.array(Y_train)
Y_test=np.array(Y_test)

In [6]:
class MyNB:
    def fit(self, X,y,alpha=1):
        self.classes=np.unique(y)
        #V=X.shape[1]
        #X=X.toarray()
        self.log_prior={}
        self.log_likelihood={}

        for c in self.classes:
            X_c=X[y==c]#Features coresponding to class c
            self.log_prior[c]=np.log(len(X_c)/len(X))
            word_counts=X_c.sum(axis=0)+alpha#Column sum
            total_words=word_counts.sum()
            self.log_likelihood[c]=np.log(word_counts/total_words)

    def predict(self,X):
        preds=[]
        for x in X:
            #x=x.toarray()
            scores={}
            for c in self.classes:
                scores[c]=self.log_prior[c]+(x*self.log_likelihood[c]).sum()
            print(scores)
            preds.append(max(scores,key=scores.get))
        return(np.array(preds))

mymod=MyNB()
mymod.fit(X_trainv,Y_train)
Ypred_test=mymod.predict(X_testv)
#print(Ypred_test)
print(f"Accuracy:{accuracy_score(Y_test,Ypred_test)}")

{np.str_('neg'): np.float64(-2330.786674390929), np.str_('pos'): np.float64(-2314.8088631512987)}
{np.str_('neg'): np.float64(-1922.6824378940908), np.str_('pos'): np.float64(-1898.751506420034)}
{np.str_('neg'): np.float64(-1136.8265941997845), np.str_('pos'): np.float64(-1145.233565391183)}
{np.str_('neg'): np.float64(-2728.3216461170605), np.str_('pos'): np.float64(-2800.3698265464063)}
{np.str_('neg'): np.float64(-854.2818544278169), np.str_('pos'): np.float64(-851.3602756776813)}
{np.str_('neg'): np.float64(-1689.9970831153164), np.str_('pos'): np.float64(-1707.1278226825063)}
{np.str_('neg'): np.float64(-2083.7227911392056), np.str_('pos'): np.float64(-2084.399294614968)}
{np.str_('neg'): np.float64(-2612.4119076468864), np.str_('pos'): np.float64(-2628.342488385201)}
{np.str_('neg'): np.float64(-4152.2607166154685), np.str_('pos'): np.float64(-4204.425269408871)}
{np.str_('neg'): np.float64(-2445.350034201962), np.str_('pos'): np.float64(-2442.4622141709706)}
{np.str_('neg'): np

In [7]:
#Logistic Regression
class mylogR:
    def fit(self, X,y,lr=0.01,epochs=200):
        y=(y=='pos').astype(int)
        self.w=np.zeros(X.shape[1])
        self.b=0

        for _ in range(epochs):
            z=X@self.w+self.b
            y_hat=1/(1+np.exp(-z))
            grad_w=X.T@(y_hat-y)/len(y)
            grad_b=np.mean(y_hat-y)
            self.w=self.w-lr*grad_w
            self.b=self.b-lr*grad_b

    def predict(self,X):
        z=X@self.w+self.b
        return(np.where(z>=0,'pos','neg'))
    
mymod=mylogR()
mymod.fit(X_trainv,Y_train)
Ypred_test=mymod.predict(X_testv)
#print(Ypred_test)
print(f"Accuracy:{accuracy_score(Y_test,Ypred_test)}")

Accuracy:0.81
